# Ventes SAP pour le reporting RSE

In [ ]:
# Fabric notebook: nb_consolidate_sap_and_csr_data
# Workspace: BM_F_D - SAP-B1 / Lakehouse: LH_CSR_Reporting
#
# À lancer juste après nb_consolidate_csr_data, dans le même pipeline.
#
# Ajoute les ventes (Env.1) aux indicateurs RSE :
#   - lit la mesure "Sales | Commercial" du modèle sémantique du dashboard
#     commercial, par entité SAP, année et mois, scénario 1 (réel) : même
#     chiffre que dans le rapport commercial ;
#   - regroupe les entités SAP par BU (ENTITY_TO_BU) ;
#   - ajoute ces ventes en lignes ID = "Env.1" à CSR_indicators_report et
#     écrit le tout dans la table Delta CSR_gold_reporting (format long),
#     lue par le tableau de bord.
#
# Aucun indicateur n'est calculé ici : les ratios (ex. Wat.2 = Wat.1 / Env.1)
# sont calculés à la volée dans le tableau de bord, au niveau d'agrégation
# affiché.
#
# Versionné dans le dépôt GitHub CSR_reporting (scripts/), exporté depuis
# Fabric sans les sorties (scripts/import_fabric_notebook.py).

In [ ]:
import pandas as pd
import sempy.fabric as fabric

In [ ]:
# Paramètres
WS = "BM_F_D - SAP-B1"
DS = "Bioline Agrosciences sales - Copy"  # modèle sémantique du dashboard commercial
SALES_MEASURE = "Sales | Commercial"
FIRST_YEAR = 2026  # première année du reporting RSE
OUTPUT_TABLE = "CSR_gold_reporting"

MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

# Entité SAP (d_Entity[Entity]) -> BU du reporting RSE. Plusieurs entités
# peuvent aller dans la même BU : leurs ventes s'additionnent.
ENTITY_TO_BU = {
    "BIOLINE AGROSCIENCES FRANCE": "BFR",
    "VIRIDAXIS": "Viridaxis",
    "BIOLINE UK": "BUK",
    "DUDUTECH IPM LTD": "BAF",
    "DUDUTECH PROPERTIES LTD": "BAF",
    "BIOLINE IBERIA": "BIB",
    "BIOLINE USA": "BUS",
    "BIOLINE AGROSCIENCES MEXICANA": "BUS",
}

## Ventes, depuis le modèle sémantique du dashboard commercial

In [ ]:
df_sales = fabric.evaluate_measure(
    dataset=DS,
    workspace=WS,
    measure=[SALES_MEASURE],
    groupby_columns=["d_Entity[Entity]", "d_Calendar[Year]", "d_Calendar[Month]"],
    filters={"d_Scenary[Cod_Scenary]": ["1"], "d_Calendar[Year]": list(range(FIRST_YEAR, 2101))},
)

In [ ]:
# Entité -> BU, numéro de mois -> nom du mois (comme dans CSR_indicators_report)
sales = pd.DataFrame(df_sales).rename(columns={SALES_MEASURE: "Value"})
sales["BU"] = sales["Entity"].map(ENTITY_TO_BU)
unmapped = sales.loc[sales["BU"].isna(), "Entity"].unique()
if len(unmapped):
    print(f"WARNING: entité(s) SAP sans BU, ignorées (à ajouter dans ENTITY_TO_BU) : {list(unmapped)}")

env1 = sales.dropna(subset=["BU"]).groupby(["BU", "Year", "Month"], as_index=False)["Value"].sum()
env1["Year"] = env1["Year"].astype(int)
env1["Month"] = env1["Month"].astype(int).map(lambda m: MONTHS[m - 1])
env1["Value"] = env1["Value"].astype(float)
env1["ID"] = "Env.1"

# Contrôle visuel : ventes en k€ par BU et par mois
print((env1.pivot_table(index="BU", columns="Month", values="Value", aggfunc="sum")
       .reindex(columns=[m for m in MONTHS if m in set(env1["Month"])]) / 1000).round(0).to_string())

## Indicateurs RSE + ventes

In [ ]:
indicators = spark.read.table("CSR_indicators_report").toPandas()[["BU", "Year", "Month", "ID", "Value"]]

gold = pd.concat([indicators, env1[["BU", "Year", "Month", "ID", "Value"]]], ignore_index=True)
gold = gold.dropna(subset=["Value"])
gold["Year"] = gold["Year"].astype(int)
gold["Value"] = gold["Value"].astype(float)
gold["MonthNumber"] = gold["Month"].map(lambda m: MONTHS.index(m) + 1)
gold["MonthDate"] = pd.to_datetime(dict(year=gold["Year"], month=gold["MonthNumber"], day=1))
gold = gold[["BU", "Year", "Month", "MonthNumber", "MonthDate", "ID", "Value"]]
print(f"{len(gold)} rows ({(gold.ID == 'Env.1').sum()} Env.1).")

## Écriture

In [ ]:
(spark.createDataFrame(gold)
 .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(OUTPUT_TABLE))
print(f"{OUTPUT_TABLE}: {len(gold)} row(s) written.")